# Canonical Shape Hash and Waveform Reuse in qxpulse

This notebook explains waveform reuse with two pulse types:

- **Predefined waveform** created with `Gaussian(...)`
- **Custom waveform** created with `Pulse(values=...)`

In both cases, qxpulse uses `shape_hash` to identify the canonical waveform shape, while `scale` and `phase` stay as per-event runtime factors.

In [ ]:
import numpy as np

from qubex.pulse import Gaussian, Pulse

## 1) Predefined waveform with `Gaussian`

Two Gaussian pulses that differ only by `scale`/`phase` share the same canonical shape hash.

In [ ]:
gaussian_base = Gaussian(
    duration=40.0,
    amplitude=1.0,
    sigma=8.0,
    sampling_period=0.5,
)
scaled_and_shifted = gaussian_base.scaled(0.6).shifted(-np.pi / 6)

gaussian_base.plot(title="Gaussian base")
scaled_and_shifted.plot(title="Gaussian scaled and shifted")

print("gaussian base scale, phase:", gaussian_base.scale, gaussian_base.phase)
print(
    "scaled_and_shifted scale, phase:",
    scaled_and_shifted.scale,
    scaled_and_shifted.phase,
)
print(
    "gaussian shape_values equal:",
    np.allclose(gaussian_base.shape_values, scaled_and_shifted.shape_values),
)
print(
    "gaussian shape_hash equal:",
    gaussian_base.shape_hash == scaled_and_shifted.shape_hash,
)

In [ ]:
gaussian_other = Gaussian(
    duration=40.0,
    amplitude=1.0,
    sigma=10.0,
    sampling_period=0.5,
)
print(
    "different Gaussian parameters -> different shape_hash:",
    gaussian_base.shape_hash != gaussian_other.shape_hash,
)

## 2) Custom waveform with `Pulse`

The same rule applies to a manually defined arbitrary waveform.

In [ ]:
custom_base = Pulse(
    values=np.array([0.0, 1.0, 0.5, 0.0], dtype=np.complex128),
    sampling_period=0.5,
)
custom_variant = custom_base.scaled(0.4).shifted(np.pi / 3)

custom_base.plot(title="Custom waveform base")
custom_variant.plot(title="Custom waveform scaled and shifted")

print("custom base scale, phase:", custom_base.scale, custom_base.phase)
print("custom variant scale, phase:", custom_variant.scale, custom_variant.phase)
print(
    "custom shape_values equal:",
    np.allclose(custom_base.shape_values, custom_variant.shape_values),
)
print("custom shape_hash equal:", custom_base.shape_hash == custom_variant.shape_hash)

## 3) Minimal registration simulation (mixed pulse types)

This emulates QuEL-3-style registration:

- Register one waveform per unique `shape_hash`
- Keep `gain` and `phase_offset_deg` in each event

In [ ]:
events_input = [
    gaussian_base,
    scaled_and_shifted,
    gaussian_other,
    custom_base,
    custom_variant,
]

waveform_name_by_shape_hash = {}
events = []

for pulse in events_input:
    shape_hash = pulse.shape_hash
    waveform_name = waveform_name_by_shape_hash.get(shape_hash)
    if waveform_name is None:
        waveform_name = f"waveform_{len(waveform_name_by_shape_hash):04d}"
        waveform_name_by_shape_hash[shape_hash] = waveform_name

    events.append(
        {
            "pulse_type": type(pulse).__name__,
            "waveform_name": waveform_name,
            "gain": pulse.scale,
            "phase_offset_deg": float(np.rad2deg(pulse.phase)),
        }
    )

print("registered waveforms:", len(waveform_name_by_shape_hash))
print("events:", len(events))
for event in events:
    print(event)

Result: even with mixed pulse types (`Gaussian` and `Pulse`), scale/phase variants are reused by canonical shape hash, reducing waveform registrations.